<a href="https://colab.research.google.com/github/yhshengjy/ClinPKPD/blob/main/Notebook3_%E8%82%BE%E5%8A%9F%E8%83%BD%E5%8F%98%E5%8C%96%E4%B8%8E%E5%89%82%E9%87%8F%E8%B0%83%E6%95%B4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3：肾功能变化与剂量调整

本 Notebook 是临床药学 PK/PD 交互式模拟平台的第三个模块。

前两个 Notebook 已经学习了：

- 单次给药后的血药浓度变化
- 口服和静脉给药的一室模型
- 多剂量给药与稳态浓度
- 治疗窗和基础 Emax 药效模型

本节进一步把 PK/PD 模型和患者特征联系起来，重点讨论：

> 肾功能下降时，药物清除率如何改变？  
> 清除率改变后，血药浓度、AUC、半衰期和毒性风险如何变化？  
> 如何用模型思维理解剂量调整？

本 Notebook 的核心逻辑是：

$$
Patient\ renal\ function \rightarrow Drug\ clearance \rightarrow Concentration(t) \rightarrow Exposure \rightarrow Dose\ adjustment
$$

请注意：本 Notebook 用于教学模拟，不替代真实临床处方。真实患者用药调整必须结合药品说明书、医院指南、肾替代治疗方式、感染严重程度、TDM 结果和临床反应。

## 1. 学习目标

完成本 Notebook 后，你应该能够：

1. 解释为什么肾功能会影响药物清除率和全身暴露量。
2. 使用 Cockcroft-Gault 公式估算肌酐清除率。
3. 区分 CrCl、eGFR 及其在药物剂量调整中的不同作用。
4. 描述肾功能下降如何影响 AUC、半衰期、峰浓度、谷浓度和蓄积风险。
5. 比较常见的肾功能剂量调整策略，包括降低剂量和延长给药间隔。

## 2. 临床情境导入

某患者因感染需要使用一种主要经肾脏清除的抗菌药物。

标准给药方案为：

$$
500\ mg\ q24h
$$

但患者年龄较大，血清肌酐升高，估算肌酐清除率下降。

这时临床药师需要思考：

- 患者的肾功能大约是多少？
- 药物清除率是否会下降？
- 如果仍使用标准剂量，是否会出现蓄积？
- 应该减少每次剂量，还是延长给药间隔？
- 如何从浓度-时间曲线判断调整方案是否更合理？

本 Notebook 将通过交互模拟逐步回答这些问题。

## 3. 肾功能与药物清除率

药物总清除率可以简化为：

$$
CL_{total} = CL_{renal} + CL_{nonrenal}
$$

其中：

| 符号 | 含义 |
|---|---|
| $CL_{renal}$ | 肾清除率 |
| $CL_{nonrenal}$ | 非肾清除率，例如肝代谢、胆汁排泄等 |
| $CL_{total}$ | 总清除率 |

对于主要经肾脏清除的药物，肾功能下降会使总清除率明显下降。  
对于主要经肝脏代谢的药物，肾功能下降对总清除率的影响可能较小。

为了教学模拟，我们使用肾清除比例 $f_e$ 表示药物清除中依赖肾脏的部分：

$$
f_e = \frac{CL_{renal}}{CL_{total}}
$$

当 $f_e$ 越大，说明药物越依赖肾脏清除，肾功能下降对药物暴露的影响越明显。

## 4. 肌酐清除率 CrCl 的估算

临床上常使用 Cockcroft-Gault 公式估算肌酐清除率：

$$
CrCl = \frac{(140 - Age) \times Weight}{72 \times SCr}
$$

女性患者通常乘以 0.85：

$$
CrCl_{female} = CrCl \times 0.85
$$

其中：

| 符号 | 含义 | 单位 |
|---|---|---|
| Age | 年龄 | years |
| Weight | 体重 | kg |
| SCr | 血清肌酐 | mg/dL |
| CrCl | 肌酐清除率 | mL/min |

如果血清肌酐单位为 μmol/L，可近似换算为：

$$
SCr(mg/dL) = \frac{SCr(\mu mol/L)}{88.4}
$$

注意：Cockcroft-Gault 公式存在局限，尤其是在极端体重、急性肾损伤、肌肉量异常、老年衰弱、孕妇等情况下。真实临床中需结合患者情况和医院规范判断。

## 5. CrCl、eGFR 与剂量调整

临床检验报告常给出 eGFR，单位通常是：

$$
mL/min/1.73m^2
$$

而很多药品说明书中的剂量调整建议使用的是 CrCl，单位通常是：

$$
mL/min
$$

因此，在进行药物剂量调整时应注意：

- 药品说明书使用 CrCl 还是 eGFR。
- 公式是否按体表面积标准化。
- 患者是否为急性肾损伤或肾功能快速变化。
- 是否接受血液透析、腹膜透析或 CRRT。

本 Notebook 使用 Cockcroft-Gault 公式估算 CrCl，主要用于教学模拟。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True


def scr_umol_l_to_mg_dl(scr_umol_l):
    """
    Convert serum creatinine from micromol/L to mg/dL.
    """
    return scr_umol_l / 88.4


def height_cm_to_inches(height_cm):
    """
    Convert height from cm to inches.
    """
    return height_cm / 2.54


def calculate_ibw(sex, height_cm):
    """
    Calculate ideal body weight using Devine-style equations.
    This is used here only for teaching simulation.
    """
    height_in = height_cm_to_inches(height_cm)
    inches_over_5ft = max(height_in - 60, 0)

    if sex == "Male":
        ibw = 50 + 2.3 * inches_over_5ft
    else:
        ibw = 45.5 + 2.3 * inches_over_5ft

    return ibw


def choose_weight_for_cg(sex, height_cm, actual_weight_kg, method):
    """
    Choose body weight for Cockcroft-Gault calculation.
    """
    ibw = calculate_ibw(sex, height_cm)
    adjusted_bw = ibw + 0.4 * (actual_weight_kg - ibw)

    if method == "Actual body weight":
        selected_weight = actual_weight_kg
    elif method == "Ideal body weight":
        selected_weight = ibw
    else:
        selected_weight = adjusted_bw

    return selected_weight, ibw, adjusted_bw


def cockcroft_gault_crcl(age_years, sex, weight_kg, scr_mg_dl):
    """
    Estimate creatinine clearance using Cockcroft-Gault equation.
    """
    crcl = ((140 - age_years) * weight_kg) / (72 * scr_mg_dl)

    if sex == "Female":
        crcl *= 0.85

    return crcl


def classify_crcl(crcl_ml_min):
    """
    Simple renal function category for medication dosing discussion.
    """
    if crcl_ml_min >= 90:
        return "Normal or near normal"
    elif crcl_ml_min >= 60:
        return "Mild decrease"
    elif crcl_ml_min >= 30:
        return "Moderate decrease"
    elif crcl_ml_min >= 15:
        return "Severe decrease"
    else:
        return "Kidney failure range"

## 6. 交互模拟 1：估算患者 CrCl

下面通过患者年龄、性别、体重、身高和血清肌酐估算 CrCl。

你可以选择血清肌酐单位：

- mg/dL
- μmol/L

也可以选择 Cockcroft-Gault 公式中使用的体重：

- Actual body weight：实际体重
- Ideal body weight：理想体重
- Adjusted body weight：校正体重

请观察：

- 年龄增加时 CrCl 如何变化？
- SCr 升高时 CrCl 如何变化？
- 同样 SCr 下，性别和体重如何影响 CrCl？
- 体重选择方法不同，CrCl 是否会明显不同？

In [ ]:
def plot_crcl_calculator(
    age_years=70,
    sex="Male",
    actual_weight_kg=70,
    height_cm=170,
    scr_value=1.5,
    scr_unit="mg/dL",
    weight_method="Actual body weight"
):
    if scr_unit == "mg/dL":
        scr_mg_dl = scr_value
    else:
        scr_mg_dl = scr_umol_l_to_mg_dl(scr_value)

    selected_weight, ibw, adjusted_bw = choose_weight_for_cg(
        sex=sex,
        height_cm=height_cm,
        actual_weight_kg=actual_weight_kg,
        method=weight_method
    )

    crcl = cockcroft_gault_crcl(
        age_years=age_years,
        sex=sex,
        weight_kg=selected_weight,
        scr_mg_dl=scr_mg_dl
    )

    category = classify_crcl(crcl)

    summary = pd.DataFrame({
        "Parameter": [
            "Age",
            "Sex",
            "Height",
            "Actual body weight",
            "Ideal body weight",
            "Adjusted body weight",
            "Selected weight method",
            "Selected weight",
            "SCr",
            "Estimated CrCl",
            "Renal function category"
        ],
        "Value": [
            f"{age_years:.0f} years",
            sex,
            f"{height_cm:.1f} cm",
            f"{actual_weight_kg:.1f} kg",
            f"{ibw:.1f} kg",
            f"{adjusted_bw:.1f} kg",
            weight_method,
            f"{selected_weight:.1f} kg",
            f"{scr_mg_dl:.2f} mg/dL",
            f"{crcl:.1f} mL/min",
            category
        ]
    })

    display(summary)


interact(
    plot_crcl_calculator,
    age_years=IntSlider(value=70, min=18, max=95, step=1, description="Age"),
    sex=Dropdown(options=["Male", "Female"], value="Male", description="Sex"),
    actual_weight_kg=FloatSlider(value=70, min=35, max=150, step=1, description="Weight"),
    height_cm=FloatSlider(value=170, min=140, max=200, step=1, description="Height"),
    scr_value=FloatSlider(value=1.5, min=0.4, max=8.0, step=0.1, description="SCr"),
    scr_unit=Dropdown(options=["mg/dL", "umol/L"], value="mg/dL", description="SCr unit"),
    weight_method=Dropdown(
        options=["Actual body weight", "Ideal body weight", "Adjusted body weight"],
        value="Actual body weight",
        description="Weight"
    )
);

interactive(children=(IntSlider(value=70, description='Age', max=95, min=18), Dropdown(description='Sex', opti…

## 7. 观察任务 1：CrCl 的影响因素

请完成以下操作：

### 任务 A：标准患者

设置：

- Age = 70 years
- Sex = Male
- Weight = 70 kg
- Height = 170 cm
- SCr = 1.5 mg/dL
- Weight method = Actual body weight

记录：

- Estimated CrCl
- Renal function category

### 任务 B：只改变年龄

将 Age 改为 40 years。

观察：

- CrCl 是否升高？
- 为什么同样 SCr 下，年龄会影响 CrCl？

### 任务 C：只改变 SCr

将 Age 恢复为 70 years，然后将 SCr 改为 3.0 mg/dL。

观察：

- CrCl 是否下降？
- 肾功能分类是否改变？

### 任务 D：改变体重选择方法

保持其他参数不变，分别选择：

- Actual body weight
- Ideal body weight
- Adjusted body weight

观察：

- CrCl 是否发生变化？
- 这说明剂量调整时体重选择可能带来什么影响？

## 8. 从肾功能到药物清除率

为了把患者肾功能和药物清除率联系起来，本 Notebook 使用一个简化模型：

$$
CL_{patient} = CL_{normal} \times \left[(1-f_e) + f_e \times \frac{CrCl_{patient}}{CrCl_{normal}}\right]
$$

其中：

| 符号 | 含义 |
|---|---|
| $CL_{patient}$ | 患者当前总清除率 |
| $CL_{normal}$ | 肾功能正常时的总清除率 |
| $f_e$ | 肾清除比例 |
| $CrCl_{patient}$ | 患者估算肌酐清除率 |
| $CrCl_{normal}$ | 参考正常肌酐清除率，本节默认为 100 mL/min |

当 $f_e = 1$ 时，表示药物完全依赖肾脏清除。  
当 $f_e = 0$ 时，表示药物清除基本不依赖肾脏。

因此，肾功能下降对不同药物的影响不同。

In [ ]:
def estimate_patient_clearance(cl_normal_l_h, crcl_patient_ml_min, fe_renal, crcl_normal_ml_min=100):
    """
    Estimate patient clearance based on renal function and fraction excreted renally.
    """
    renal_function_ratio = crcl_patient_ml_min / crcl_normal_ml_min
    clearance_ratio = (1 - fe_renal) + fe_renal * renal_function_ratio
    clearance_ratio = max(clearance_ratio, 0.02)
    cl_patient_l_h = cl_normal_l_h * clearance_ratio

    return cl_patient_l_h, clearance_ratio


def one_compartment_iv_bolus(t, dose_mg, vd_l, cl_l_h):
    """
    One-compartment IV bolus model.
    """
    k_elim = cl_l_h / vd_l
    concentration = (dose_mg / vd_l) * np.exp(-k_elim * t)
    auc = dose_mg / cl_l_h
    half_life = np.log(2) / k_elim

    return concentration, k_elim, half_life, auc


def one_compartment_oral(t, dose_mg, vd_l, cl_l_h, ka_h, bioavailability):
    """
    One-compartment oral model with first-order absorption.
    """
    k_elim = cl_l_h / vd_l

    if np.isclose(ka_h, k_elim):
        concentration = bioavailability * dose_mg / vd_l * k_elim * t * np.exp(-k_elim * t)
    else:
        concentration = (
            bioavailability * dose_mg * ka_h / (vd_l * (ka_h - k_elim))
            * (np.exp(-k_elim * t) - np.exp(-ka_h * t))
        )

    concentration = np.maximum(concentration, 0)
    auc = bioavailability * dose_mg / cl_l_h
    half_life = np.log(2) / k_elim

    return concentration, k_elim, half_life, auc


def calculate_basic_pk_metrics(t, concentration, auc, half_life):
    """
    Calculate basic PK metrics.
    """
    cmax = np.max(concentration)
    tmax = t[np.argmax(concentration)]

    return {
        "Cmax": cmax,
        "Tmax": tmax,
        "AUC": auc,
        "Half-life": half_life
    }

## 9. 交互模拟 2：肾功能下降如何改变单次给药曲线？

下面模拟一个药物在不同肾功能患者体内的浓度变化。

你可以调节：

- CrCl：患者肌酐清除率
- $f_e$：药物肾清除比例
- CL normal：肾功能正常时的清除率
- Vd：分布容积
- Route：静脉或口服给药

请观察：

- CrCl 降低时，CL patient 是否下降？
- 半衰期是否延长？
- AUC 是否升高？
- $f_e$ 越大，肾功能下降的影响是否越明显？

In [ ]:
def plot_renal_function_single_dose(
    route="Oral",
    dose_mg=500,
    vd_l=70,
    cl_normal_l_h=8,
    crcl_patient_ml_min=40,
    fe_renal=0.8,
    ka_h=1.2,
    bioavailability=0.9,
    t_end_h=72
):
    t = np.linspace(0, t_end_h, 1000)

    cl_patient_l_h, clearance_ratio = estimate_patient_clearance(
        cl_normal_l_h=cl_normal_l_h,
        crcl_patient_ml_min=crcl_patient_ml_min,
        fe_renal=fe_renal
    )

    if route == "IV bolus":
        conc_normal, _, half_life_normal, auc_normal = one_compartment_iv_bolus(
            t=t, dose_mg=dose_mg, vd_l=vd_l, cl_l_h=cl_normal_l_h
        )
        conc_patient, _, half_life_patient, auc_patient = one_compartment_iv_bolus(
            t=t, dose_mg=dose_mg, vd_l=vd_l, cl_l_h=cl_patient_l_h
        )
    else:
        conc_normal, _, half_life_normal, auc_normal = one_compartment_oral(
            t=t, dose_mg=dose_mg, vd_l=vd_l, cl_l_h=cl_normal_l_h,
            ka_h=ka_h, bioavailability=bioavailability
        )
        conc_patient, _, half_life_patient, auc_patient = one_compartment_oral(
            t=t, dose_mg=dose_mg, vd_l=vd_l, cl_l_h=cl_patient_l_h,
            ka_h=ka_h, bioavailability=bioavailability
        )

    metrics_normal = calculate_basic_pk_metrics(t, conc_normal, auc_normal, half_life_normal)
    metrics_patient = calculate_basic_pk_metrics(t, conc_patient, auc_patient, half_life_patient)

    fig, ax = plt.subplots()
    ax.plot(t, conc_normal, linewidth=2, label="Normal renal function")
    ax.plot(t, conc_patient, linewidth=2, linestyle="--", label="Patient renal function")

    ax.set_title("Effect of Renal Function on Single-Dose PK")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "Metric": [
            "Route",
            "Patient CrCl",
            "Renal fraction fe",
            "Normal CL",
            "Patient CL",
            "Clearance ratio",
            "Normal half-life",
            "Patient half-life",
            "Normal AUC",
            "Patient AUC",
            "AUC fold-change"
        ],
        "Value": [
            route,
            f"{crcl_patient_ml_min:.1f} mL/min",
            f"{fe_renal:.2f}",
            f"{cl_normal_l_h:.2f} L/h",
            f"{cl_patient_l_h:.2f} L/h",
            f"{clearance_ratio:.2f}",
            f"{metrics_normal['Half-life']:.2f} h",
            f"{metrics_patient['Half-life']:.2f} h",
            f"{metrics_normal['AUC']:.2f} mg*h/L",
            f"{metrics_patient['AUC']:.2f} mg*h/L",
            f"{metrics_patient['AUC'] / metrics_normal['AUC']:.2f}"
        ]
    })

    display(summary)


interact(
    plot_renal_function_single_dose,
    route=Dropdown(options=["IV bolus", "Oral"], value="Oral", description="Route"),
    dose_mg=FloatSlider(value=500, min=100, max=2000, step=100, description="Dose"),
    vd_l=FloatSlider(value=70, min=10, max=150, step=5, description="Vd"),
    cl_normal_l_h=FloatSlider(value=8, min=1, max=20, step=0.5, description="CL normal"),
    crcl_patient_ml_min=FloatSlider(value=40, min=5, max=120, step=5, description="CrCl"),
    fe_renal=FloatSlider(value=0.8, min=0, max=1, step=0.05, description="fe"),
    ka_h=FloatSlider(value=1.2, min=0.1, max=5, step=0.1, description="ka"),
    bioavailability=FloatSlider(value=0.9, min=0.1, max=1.0, step=0.05, description="F"),
    t_end_h=FloatSlider(value=72, min=24, max=168, step=12, description="Time")
);

interactive(children=(Dropdown(description='Route', index=1, options=('IV bolus', 'Oral'), value='Oral'), Floa…

## 10. 观察任务 2：肾功能、$f_e$ 与暴露变化

请完成以下操作：

### 任务 A：肾功能中度下降

设置：

- Route = Oral
- Dose = 500 mg
- Vd = 70 L
- CL normal = 8 L/h
- CrCl = 40 mL/min
- $f_e$ = 0.8
- F = 0.9

记录：

- Patient CL
- Patient half-life
- Patient AUC
- AUC fold-change

### 任务 B：肾功能进一步下降

将 CrCl 改为 15 mL/min。

观察：

- Patient CL 是否进一步下降？
- AUC fold-change 是否增加？
- 半衰期是否延长？

### 任务 C：改变肾清除比例

保持 CrCl = 15 mL/min，分别设置：

- $f_e$ = 0.2
- $f_e$ = 0.8
- $f_e$ = 1.0

观察：

- 哪种情况下 AUC 增加最明显？
- 这说明为什么并不是所有药物都需要同样程度的肾功能剂量调整？

## 11. 剂量调整的基本思路

肾功能下降后，药物清除率下降。为了避免过度暴露，常见剂量调整策略包括：

### 策略 1：减少每次剂量，保持给药间隔不变

例如：

$$
500\ mg\ q24h \rightarrow 250\ mg\ q24h
$$

这种方式可以降低暴露，同时维持较稳定的给药节律。

### 策略 2：保持每次剂量不变，延长给药间隔

例如：

$$
500\ mg\ q24h \rightarrow 500\ mg\ q48h
$$

这种方式可以保留较高峰浓度，但给药间隔内的谷浓度可能更低。

### 策略 3：同时减少剂量并延长间隔

例如：

$$
500\ mg\ q24h \rightarrow 250\ mg\ q48h
$$

这种方式多用于肾功能严重下降或治疗窗较窄的药物。

剂量调整的核心不是机械套公式，而是理解：

$$
Maintenance\ dose\ rate \propto CL
$$

也就是说，维持剂量速率通常应随清除率下降而下降。

In [ ]:
def concentration_after_dose(t_after_dose, dose_mg, vd_l, cl_l_h, route, ka_h, bioavailability):
    """
    Concentration contribution after a single dose.
    """
    t_after_dose = np.asarray(t_after_dose)
    concentration = np.zeros_like(t_after_dose, dtype=float)
    mask = t_after_dose >= 0
    dt = t_after_dose[mask]

    k_elim = cl_l_h / vd_l

    if route == "IV bolus":
        concentration[mask] = (dose_mg / vd_l) * np.exp(-k_elim * dt)
    else:
        if np.isclose(ka_h, k_elim):
            concentration[mask] = bioavailability * dose_mg / vd_l * k_elim * dt * np.exp(-k_elim * dt)
        else:
            concentration[mask] = (
                bioavailability * dose_mg * ka_h / (vd_l * (ka_h - k_elim))
                * (np.exp(-k_elim * dt) - np.exp(-ka_h * dt))
            )

    return np.maximum(concentration, 0)


def multiple_dose_concentration(t, dose_mg, tau_h, duration_h, vd_l, cl_l_h,
                                route="Oral", ka_h=1.2, bioavailability=0.9,
                                first_dose_mg=None):
    """
    Multiple-dose concentration profile with optional first dose.
    """
    concentration = np.zeros_like(t, dtype=float)
    dose_times = np.arange(0, duration_h + 1e-9, tau_h)

    for i, dose_time in enumerate(dose_times):
        current_dose = dose_mg
        if i == 0 and first_dose_mg is not None:
            current_dose = first_dose_mg

        concentration += concentration_after_dose(
            t_after_dose=t - dose_time,
            dose_mg=current_dose,
            vd_l=vd_l,
            cl_l_h=cl_l_h,
            route=route,
            ka_h=ka_h,
            bioavailability=bioavailability
        )

    return concentration, dose_times


def steady_state_interval_concentration(
    dose_mg, tau_h, vd_l, cl_l_h, route="Oral",
    ka_h=1.2, bioavailability=0.9, n_points=4001
):
    """
    Steady-state concentration profile over one complete dosing interval.

    The profile is calculated analytically for a one-compartment model with
    first-order elimination and, for oral dosing, first-order absorption.
    """
    t_interval = np.linspace(0, tau_h, n_points)
    k_elim = cl_l_h / vd_l

    if route == "IV bolus":
        denominator = 1 - np.exp(-k_elim * tau_h)
        concentration = (
            dose_mg / vd_l
            * np.exp(-k_elim * t_interval)
            / denominator
        )
    else:
        if np.isclose(ka_h, k_elim):
            r = np.exp(-k_elim * tau_h)
            concentration = (
                bioavailability * dose_mg / vd_l
                * k_elim * np.exp(-k_elim * t_interval)
                * (
                    t_interval / (1 - r)
                    + tau_h * r / (1 - r) ** 2
                )
            )
        else:
            concentration = (
                bioavailability * dose_mg * ka_h / (vd_l * (ka_h - k_elim))
                * (
                    np.exp(-k_elim * t_interval) / (1 - np.exp(-k_elim * tau_h))
                    - np.exp(-ka_h * t_interval) / (1 - np.exp(-ka_h * tau_h))
                )
            )

    return t_interval, np.maximum(concentration, 0)


def steady_state_interval_metrics(
    dose_mg, tau_h, vd_l, cl_l_h, mec, mtc,
    route="Oral", ka_h=1.2, bioavailability=0.9
):
    """
    Calculate steady-state exposure metrics over one complete dosing interval.
    """
    t_interval, concentration = steady_state_interval_concentration(
        dose_mg=dose_mg,
        tau_h=tau_h,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        route=route,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    cmax_ss = np.max(concentration)
    cmin_ss = np.min(concentration)

    # For linear PK, steady-state AUC over one dosing interval is Dose/CL
    # (multiplied by F for oral administration).
    f_auc = 1.0 if route == "IV bolus" else bioavailability
    auc_tau_ss = f_auc * dose_mg / cl_l_h
    cavg_ss = auc_tau_ss / tau_h

    pct_below_mec = (
        np.trapz((concentration < mec).astype(float), t_interval) / tau_h * 100
    )
    pct_above_mtc = (
        np.trapz((concentration > mtc).astype(float), t_interval) / tau_h * 100
    )

    return {
        "Cmax_ss": cmax_ss,
        "Cmin_ss": cmin_ss,
        "AUC_tau_ss": auc_tau_ss,
        "Cavg_ss": cavg_ss,
        "Pct_time_below_MEC": pct_below_mec,
        "Pct_time_above_MTC": pct_above_mtc
    }


## 12. 交互模拟 3：比较不同剂量调整策略

下面的模拟比较四种情况：

1. **Normal reference**：正常肾功能下的标准给药方案
2. **No adjustment**：肾功能下降后仍维持原剂量和给药间隔
3. **Reduce dose**：按清除率下降比例减少每次剂量，给药间隔不变
4. **Extend interval**：保持每次剂量不变，按清除率下降比例延长给药间隔

结果表采用**稳态下一个完整给药间隔**的指标，可以更公平地比较不同给药间隔的方案。

重点观察：

- **Steady-state Cmax**：峰浓度如何随调整策略改变？
- **Steady-state Cmin**：延长给药间隔是否产生更低的谷浓度？
- **AUCτ,ss**：一个完整给药间隔内的总暴露是多少？注意，当给药间隔不同，AUCτ 覆盖的时间长度也不同，因此不宜单独用 AUCτ 直接比较不同 τ 方案的平均暴露。
- **Cavg,ss**：用于比较不同给药间隔方案的平均稳态暴露是否接近正常参考。
- **Time above MTC / below MEC (% of interval)**：一个完整给药间隔中，高于或低于教学阈值的时间比例是多少？

> **说明：** MEC 和 MTC 在本 Notebook 中是用于理解浓度阈值和给药方案差异的**教学性阈值**，并不代表某一特定药物的真实治疗范围或处方建议。


In [ ]:
def plot_dose_adjustment_strategies(
    route="Oral",
    usual_dose_mg=500,
    usual_tau_h=24,
    vd_l=70,
    cl_normal_l_h=8,
    crcl_patient_ml_min=30,
    fe_renal=0.8,
    ka_h=1.2,
    bioavailability=0.9,
    mec=0.5,
    mtc=8,
    duration_days=7
):
    duration_h = duration_days * 24
    t = np.linspace(0, duration_h, 2500)

    cl_patient_l_h, clearance_ratio = estimate_patient_clearance(
        cl_normal_l_h=cl_normal_l_h,
        crcl_patient_ml_min=crcl_patient_ml_min,
        fe_renal=fe_renal
    )

    reduced_dose_mg = usual_dose_mg * clearance_ratio
    extended_tau_h = usual_tau_h / clearance_ratio
    extended_tau_h = min(max(extended_tau_h, usual_tau_h), 120)

    # Normal renal-function reference regimen
    conc_normal_reference, _ = multiple_dose_concentration(
        t=t, dose_mg=usual_dose_mg, tau_h=usual_tau_h, duration_h=duration_h,
        vd_l=vd_l, cl_l_h=cl_normal_l_h, route=route, ka_h=ka_h,
        bioavailability=bioavailability
    )

    conc_standard, _ = multiple_dose_concentration(
        t=t, dose_mg=usual_dose_mg, tau_h=usual_tau_h, duration_h=duration_h,
        vd_l=vd_l, cl_l_h=cl_patient_l_h, route=route, ka_h=ka_h,
        bioavailability=bioavailability
    )

    conc_reduced_dose, _ = multiple_dose_concentration(
        t=t, dose_mg=reduced_dose_mg, tau_h=usual_tau_h, duration_h=duration_h,
        vd_l=vd_l, cl_l_h=cl_patient_l_h, route=route, ka_h=ka_h,
        bioavailability=bioavailability
    )

    conc_extended_interval, _ = multiple_dose_concentration(
        t=t, dose_mg=usual_dose_mg, tau_h=extended_tau_h, duration_h=duration_h,
        vd_l=vd_l, cl_l_h=cl_patient_l_h, route=route, ka_h=ka_h,
        bioavailability=bioavailability
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(t, conc_normal_reference, linewidth=2, linestyle="-.", label="Normal reference")
    ax.plot(t, conc_standard, linewidth=2, label="No adjustment")
    ax.plot(t, conc_reduced_dose, linewidth=2, linestyle="--", label="Reduce dose")
    ax.plot(t, conc_extended_interval, linewidth=2, linestyle=":", label="Extend interval")
    ax.axhline(mec, linestyle="--", label=f"MEC = {mec:.1f} mg/L")
    ax.axhline(mtc, linestyle="--", label=f"MTC = {mtc:.1f} mg/L")
    ax.fill_between(t, mec, mtc, alpha=0.12, label="Illustrative therapeutic window")

    ax.set_title("Dose Adjustment Strategies in Renal Impairment")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    metrics_normal = steady_state_interval_metrics(
        dose_mg=usual_dose_mg, tau_h=usual_tau_h, vd_l=vd_l,
        cl_l_h=cl_normal_l_h, mec=mec, mtc=mtc, route=route,
        ka_h=ka_h, bioavailability=bioavailability
    )
    metrics_standard = steady_state_interval_metrics(
        dose_mg=usual_dose_mg, tau_h=usual_tau_h, vd_l=vd_l,
        cl_l_h=cl_patient_l_h, mec=mec, mtc=mtc, route=route,
        ka_h=ka_h, bioavailability=bioavailability
    )
    metrics_reduced = steady_state_interval_metrics(
        dose_mg=reduced_dose_mg, tau_h=usual_tau_h, vd_l=vd_l,
        cl_l_h=cl_patient_l_h, mec=mec, mtc=mtc, route=route,
        ka_h=ka_h, bioavailability=bioavailability
    )
    metrics_extended = steady_state_interval_metrics(
        dose_mg=usual_dose_mg, tau_h=extended_tau_h, vd_l=vd_l,
        cl_l_h=cl_patient_l_h, mec=mec, mtc=mtc, route=route,
        ka_h=ka_h, bioavailability=bioavailability
    )

    metric_names = [
        "Dose",
        "Interval",
        "Dose rate",
        "Steady-state Cmax",
        "Steady-state Cmin",
        "AUCτ,ss (per dosing interval)",
        "Cavg,ss",
        "Time below MEC (% of interval)",
        "Time above MTC (% of interval)"
    ]

    def format_column(dose, tau, metrics):
        return [
            f"{dose:.0f} mg",
            f"{tau:.1f} h",
            f"{dose / tau:.1f} mg/h",
            f"{metrics['Cmax_ss']:.2f} mg/L",
            f"{metrics['Cmin_ss']:.2f} mg/L",
            f"{metrics['AUC_tau_ss']:.2f} mg·h/L",
            f"{metrics['Cavg_ss']:.2f} mg/L",
            f"{metrics['Pct_time_below_MEC']:.1f}%",
            f"{metrics['Pct_time_above_MTC']:.1f}%"
        ]

    summary = pd.DataFrame({
        "Metric": metric_names,
        "Normal reference": format_column(usual_dose_mg, usual_tau_h, metrics_normal),
        "No adjustment": format_column(usual_dose_mg, usual_tau_h, metrics_standard),
        "Reduce dose": format_column(reduced_dose_mg, usual_tau_h, metrics_reduced),
        "Extend interval": format_column(usual_dose_mg, extended_tau_h, metrics_extended)
    })

    display(summary)

    info = pd.DataFrame({
        "Parameter": [
            "Patient CrCl",
            "Normal CL",
            "Patient CL",
            "Clearance ratio",
            "Renal fraction fe"
        ],
        "Value": [
            f"{crcl_patient_ml_min:.1f} mL/min",
            f"{cl_normal_l_h:.2f} L/h",
            f"{cl_patient_l_h:.2f} L/h",
            f"{clearance_ratio:.2f}",
            f"{fe_renal:.2f}"
        ]
    })

    display(info)


interact(
    plot_dose_adjustment_strategies,
    route=Dropdown(options=["IV bolus", "Oral"], value="Oral", description="Route"),
    usual_dose_mg=FloatSlider(value=500, min=100, max=2000, step=100, description="Dose"),
    usual_tau_h=FloatSlider(value=24, min=6, max=48, step=6, description="Tau"),
    vd_l=FloatSlider(value=70, min=10, max=150, step=5, description="Vd"),
    cl_normal_l_h=FloatSlider(value=8, min=1, max=20, step=0.5, description="CL normal"),
    crcl_patient_ml_min=FloatSlider(value=30, min=5, max=120, step=5, description="CrCl"),
    fe_renal=FloatSlider(value=0.8, min=0, max=1, step=0.05, description="fe"),
    ka_h=FloatSlider(value=1.2, min=0.1, max=5, step=0.1, description="ka"),
    bioavailability=FloatSlider(value=0.9, min=0.1, max=1.0, step=0.05, description="F"),
    mec=FloatSlider(value=0.5, min=0.1, max=5, step=0.1, description="MEC"),
    mtc=FloatSlider(value=8, min=4, max=20, step=0.5, description="MTC"),
    duration_days=IntSlider(value=7, min=3, max=14, step=1, description="Days")
);


interactive(children=(Dropdown(description='Route', index=1, options=('IV bolus', 'Oral'), value='Oral'), Floa…

In [ ]:
# ================================================================
# Notebook 3：肾功能不全患者的剂量调整策略
#
# 需要前面的Cell已经定义：
# estimate_patient_clearance()
# multiple_dose_concentration()
# steady_state_interval_metrics()
# ================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from ipywidgets import interact, Dropdown, FloatSlider, IntSlider
from google.colab import drive


# ================================================================
# 【改动1】挂载Google Drive并创建图片保存文件夹
# ================================================================
drive.mount("/content/drive", force_remount=False)

save_folder = "/content/drive/MyDrive/PKPD_Plots"
os.makedirs(save_folder, exist_ok=True)


def plot_dose_adjustment_strategies(
    route="Oral",
    usual_dose_mg=500,
    usual_tau_h=24,
    vd_l=70,
    cl_normal_l_h=8,
    crcl_patient_ml_min=30,
    fe_renal=0.8,
    ka_h=1.2,
    bioavailability=0.9,
    mec=0.5,
    mtc=8,
    duration_days=7
):
    duration_h = duration_days * 24
    t = np.linspace(0, duration_h, 2500)

    cl_patient_l_h, clearance_ratio = estimate_patient_clearance(
        cl_normal_l_h=cl_normal_l_h,
        crcl_patient_ml_min=crcl_patient_ml_min,
        fe_renal=fe_renal
    )

    reduced_dose_mg = usual_dose_mg * clearance_ratio

    extended_tau_h = usual_tau_h / clearance_ratio
    extended_tau_h = min(
        max(extended_tau_h, usual_tau_h),
        120
    )

    # 正常肾功能参考方案
    conc_normal_reference, _ = multiple_dose_concentration(
        t=t,
        dose_mg=usual_dose_mg,
        tau_h=usual_tau_h,
        duration_h=duration_h,
        vd_l=vd_l,
        cl_l_h=cl_normal_l_h,
        route=route,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    # 肾功能下降，但不调整给药方案
    conc_standard, _ = multiple_dose_concentration(
        t=t,
        dose_mg=usual_dose_mg,
        tau_h=usual_tau_h,
        duration_h=duration_h,
        vd_l=vd_l,
        cl_l_h=cl_patient_l_h,
        route=route,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    # 减少剂量
    conc_reduced_dose, _ = multiple_dose_concentration(
        t=t,
        dose_mg=reduced_dose_mg,
        tau_h=usual_tau_h,
        duration_h=duration_h,
        vd_l=vd_l,
        cl_l_h=cl_patient_l_h,
        route=route,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    # 延长给药间隔
    conc_extended_interval, _ = multiple_dose_concentration(
        t=t,
        dose_mg=usual_dose_mg,
        tau_h=extended_tau_h,
        duration_h=duration_h,
        vd_l=vd_l,
        cl_l_h=cl_patient_l_h,
        route=route,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    # ============================================================
    # 绘制浓度曲线
    # 保留原Notebook的尺寸、配色、线型和图例
    # ================================================================
    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(
        t,
        conc_normal_reference,
        linewidth=2,
        linestyle="-.",
        label="Normal reference"
    )

    ax.plot(
        t,
        conc_standard,
        linewidth=2,
        label="No adjustment"
    )

    ax.plot(
        t,
        conc_reduced_dose,
        linewidth=2,
        linestyle="--",
        label="Reduce dose"
    )

    ax.plot(
        t,
        conc_extended_interval,
        linewidth=2,
        linestyle=":",
        label="Extend interval"
    )

    ax.axhline(
        mec,
        linestyle="--",
        label=f"MEC = {mec:.1f} mg/L"
    )

    ax.axhline(
        mtc,
        linestyle="--",
        label=f"MTC = {mtc:.1f} mg/L"
    )

    ax.fill_between(
        t,
        mec,
        mtc,
        alpha=0.12,
        label="Illustrative therapeutic window"
    )

    ax.set_title(
        "Dose Adjustment Strategies in Renal Impairment"
    )
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()

    # ============================================================
    # 【改动2】将当前滑块参数对应的主图保存为300 dpi
    # ================================================================
    plot_png_path = os.path.join(
        save_folder,
        "notebook3_dose_adjustment_strategies_300dpi.png"
    )

    fig.savefig(
        plot_png_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
        pad_inches=0.05
    )

    # 同时保存矢量PDF
    plot_pdf_path = os.path.join(
        save_folder,
        "notebook3_dose_adjustment_strategies_vector.pdf"
    )

    fig.savefig(
        plot_pdf_path,
        bbox_inches="tight",
        facecolor="white",
        pad_inches=0.05
    )

    plt.show()
    plt.close(fig)

    # ============================================================
    # 计算各方案的稳态指标
    # ================================================================
    metrics_normal = steady_state_interval_metrics(
        dose_mg=usual_dose_mg,
        tau_h=usual_tau_h,
        vd_l=vd_l,
        cl_l_h=cl_normal_l_h,
        mec=mec,
        mtc=mtc,
        route=route,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    metrics_standard = steady_state_interval_metrics(
        dose_mg=usual_dose_mg,
        tau_h=usual_tau_h,
        vd_l=vd_l,
        cl_l_h=cl_patient_l_h,
        mec=mec,
        mtc=mtc,
        route=route,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    metrics_reduced = steady_state_interval_metrics(
        dose_mg=reduced_dose_mg,
        tau_h=usual_tau_h,
        vd_l=vd_l,
        cl_l_h=cl_patient_l_h,
        mec=mec,
        mtc=mtc,
        route=route,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    metrics_extended = steady_state_interval_metrics(
        dose_mg=usual_dose_mg,
        tau_h=extended_tau_h,
        vd_l=vd_l,
        cl_l_h=cl_patient_l_h,
        mec=mec,
        mtc=mtc,
        route=route,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    metric_names = [
        "Dose",
        "Interval",
        "Dose rate",
        "Steady-state Cmax",
        "Steady-state Cmin",
        "AUCτ,ss (per dosing interval)",
        "Cavg,ss",
        "Time below MEC (% of interval)",
        "Time above MTC (% of interval)"
    ]

    def format_column(dose, tau, metrics):
        return [
            f"{dose:.0f} mg",
            f"{tau:.1f} h",
            f"{dose / tau:.1f} mg/h",
            f"{metrics['Cmax_ss']:.2f} mg/L",
            f"{metrics['Cmin_ss']:.2f} mg/L",
            f"{metrics['AUC_tau_ss']:.2f} mg·h/L",
            f"{metrics['Cavg_ss']:.2f} mg/L",
            f"{metrics['Pct_time_below_MEC']:.1f}%",
            f"{metrics['Pct_time_above_MTC']:.1f}%"
        ]

    summary = pd.DataFrame({
        "Metric": metric_names,

        "Normal reference": format_column(
            usual_dose_mg,
            usual_tau_h,
            metrics_normal
        ),

        "No adjustment": format_column(
            usual_dose_mg,
            usual_tau_h,
            metrics_standard
        ),

        "Reduce dose": format_column(
            reduced_dose_mg,
            usual_tau_h,
            metrics_reduced
        ),

        "Extend interval": format_column(
            usual_dose_mg,
            extended_tau_h,
            metrics_extended
        )
    })

    # ============================================================
    # 【改动3】将summary DataFrame绘制为统一样式的图片
    #
    # 样式：
    # - 不显示DataFrame索引
    # - 浅灰色粗体表头
    # - 白色和浅灰色交替行
    # - 浅灰色完整边框
    # ================================================================
    fig_table, ax_table = plt.subplots(
        figsize=(14, 5.3)
    )

    fig_table.patch.set_facecolor("white")
    ax_table.set_facecolor("white")
    ax_table.axis("off")

    table = ax_table.table(
        cellText=summary.values,
        colLabels=summary.columns,
        cellLoc="center",
        colLoc="center",

        # Metric列较宽，其余四列宽度相同
        colWidths=[
            0.32,
            0.17,
            0.17,
            0.17,
            0.17
        ],

        loc="center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9.5)
    table.scale(1, 1.55)

    # 设置表头、交替行和边框
    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor("#BFBFBF")
        cell.set_linewidth(0.6)
        cell.PAD = 0.035

        if row == 0:
            # 表头
            cell.set_facecolor("#E6E6E6")
            cell.set_text_props(
                weight="bold",
                color="black",
                ha="center",
                va="center"
            )

        elif row % 2 == 0:
            # 偶数数据行
            cell.set_facecolor("#F5F5F5")
            cell.set_text_props(
                color="black",
                ha="center",
                va="center"
            )

        else:
            # 奇数数据行
            cell.set_facecolor("white")
            cell.set_text_props(
                color="black",
                ha="center",
                va="center"
            )

    fig_table.tight_layout(pad=0.05)

    # ============================================================
    # 【改动4】将summary表格保存为300 dpi
    # ================================================================
    summary_png_path = os.path.join(
        save_folder,
        "notebook3_dose_adjustment_strategies_summary_300dpi.png"
    )

    fig_table.savefig(
        summary_png_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
        pad_inches=0.05
    )

    # 同时保存矢量PDF
    summary_pdf_path = os.path.join(
        save_folder,
        "notebook3_dose_adjustment_strategies_summary_vector.pdf"
    )

    fig_table.savefig(
        summary_pdf_path,
        bbox_inches="tight",
        facecolor="white",
        pad_inches=0.05
    )

    plt.show()
    plt.close(fig_table)

    # ============================================================
    # info仍按照原Notebook方式显示
    # ================================================================
    info = pd.DataFrame({
        "Parameter": [
            "Patient CrCl",
            "Normal CL",
            "Patient CL",
            "Clearance ratio",
            "Renal fraction fe"
        ],

        "Value": [
            f"{crcl_patient_ml_min:.1f} mL/min",
            f"{cl_normal_l_h:.2f} L/h",
            f"{cl_patient_l_h:.2f} L/h",
            f"{clearance_ratio:.2f}",
            f"{fe_renal:.2f}"
        ]
    })

    display(info)

    print("文件已保存至Google Drive：")
    print(plot_png_path)
    print(summary_png_path)


# ================================================================
# 交互式滑块
#
# 【改动5】加入continuous_update=False
# 松开滑块后才重新绘图和保存，避免拖动时反复生成文件
# ================================================================
interact(
    plot_dose_adjustment_strategies,

    route=Dropdown(
        options=["IV bolus", "Oral"],
        value="Oral",
        description="Route"
    ),

    usual_dose_mg=FloatSlider(
        value=500,
        min=100,
        max=2000,
        step=100,
        description="Dose",
        continuous_update=False
    ),

    usual_tau_h=FloatSlider(
        value=24,
        min=6,
        max=48,
        step=6,
        description="Tau",
        continuous_update=False
    ),

    vd_l=FloatSlider(
        value=70,
        min=10,
        max=150,
        step=5,
        description="Vd",
        continuous_update=False
    ),

    cl_normal_l_h=FloatSlider(
        value=8,
        min=1,
        max=20,
        step=0.5,
        description="CL normal",
        continuous_update=False
    ),

    crcl_patient_ml_min=FloatSlider(
        value=30,
        min=5,
        max=120,
        step=5,
        description="CrCl",
        continuous_update=False
    ),

    fe_renal=FloatSlider(
        value=0.8,
        min=0,
        max=1,
        step=0.05,
        description="fe",
        continuous_update=False
    ),

    ka_h=FloatSlider(
        value=1.2,
        min=0.1,
        max=5,
        step=0.1,
        description="ka",
        continuous_update=False
    ),

    bioavailability=FloatSlider(
        value=0.9,
        min=0.1,
        max=1.0,
        step=0.05,
        description="F",
        continuous_update=False
    ),

    mec=FloatSlider(
        value=0.5,
        min=0.1,
        max=5,
        step=0.1,
        description="MEC",
        continuous_update=False
    ),

    mtc=FloatSlider(
        value=8,
        min=4,
        max=20,
        step=0.5,
        description="MTC",
        continuous_update=False
    ),

    duration_days=IntSlider(
        value=7,
        min=3,
        max=14,
        step=1,
        description="Days",
        continuous_update=False
    )
);

Mounted at /content/drive


interactive(children=(Dropdown(description='Route', index=1, options=('IV bolus', 'Oral'), value='Oral'), Floa…

## 13. 观察任务 3：减量 vs 延长间隔

本任务比较肾功能下降后两种常见调整策略，并以正常肾功能下的标准方案作为参考。

### 任务 A：中度肾功能下降

设置：

- Route = Oral
- Dose = 500 mg
- Tau = 24 h
- Vd = 70 L
- CL normal = 8 L/h
- CrCl = 30 mL/min
- $f_e$ = 0.8
- MEC = 0.5 mg/L
- MTC = 8 mg/L

比较：

- Normal reference
- No adjustment
- Reduce dose
- Extend interval

记录并比较：

- Steady-state Cmax
- Steady-state Cmin
- AUCτ,ss
- Cavg,ss
- Time below MEC (% of dosing interval)
- Time above MTC (% of dosing interval)

观察：

- 肾功能下降后，如果不调整剂量，Cavg,ss 和 AUC 是否明显高于正常参考？
- Reduce dose 和 Extend interval 是否都能使**平均稳态暴露**接近正常参考？
- Reduce dose 是否产生较低的峰浓度和相对较小的峰谷波动？
- Extend interval 是否保留相对较高的峰浓度，但产生更低的谷浓度和更大的峰谷波动？

### 任务 B：肾功能严重下降

将：

- CrCl = 10 mL/min

再次比较四种情况。

观察：

- No adjustment 的 Cmax、Cmin 和 Cavg,ss 是否进一步升高？
- Time above MTC (% of dosing interval) 是否增加？
- Reduce dose 与 Extend interval 的平均稳态暴露是否仍可接近 Normal reference？
- 两种调整方式的 Cmax 和 Cmin 为什么仍然明显不同？

### 任务 C：思考如何选择调整策略

思考：

- 为什么相似的平均稳态暴露并不意味着相似的峰浓度和谷浓度？
- 对于疗效与较高峰浓度密切相关的药物，为什么延长给药间隔可能更合适？
- 对于需要减少浓度波动或维持较稳定浓度的药物，为什么减少每次剂量可能更合适？
- 对于治疗窗较窄的药物，为什么通常还需要结合 TDM 进一步个体化调整？

> **核心概念：** 肾功能剂量调整的目标不是简单地使浓度始终位于某个固定范围内，而是根据清除率变化恢复适当的药物暴露，并结合药物的 PK/PD 特征选择合适的剂量和给药间隔。


## 14. 负荷剂量与维持剂量

肾功能调整中，一个常见误区是：

> 只要肾功能下降，所有剂量都应该同比例减少。

实际上，负荷剂量和维持剂量的决定因素不同。

### 负荷剂量 Loading dose

负荷剂量主要用于快速达到目标浓度：

$$
Loading\ Dose = \frac{Target\ Concentration \times V_d}{F}
$$

因此，负荷剂量主要受以下因素影响：

- 目标浓度
- 分布容积 Vd
- 生物利用度 F

### 维持剂量 Maintenance dose

维持剂量主要用于补偿药物清除：

$$
Maintenance\ Dose\ Rate = \frac{Target\ Concentration \times CL}{F}
$$

因此，维持剂量主要受以下因素影响：

- 目标浓度
- 清除率 CL
- 生物利用度 F
- 给药间隔

肾功能下降通常首先影响 CL，因此更直接影响维持剂量和给药间隔。

In [ ]:
def calculate_loading_and_maintenance(
    target_concentration=5,
    vd_l=70,
    bioavailability=0.9,
    cl_l_h=8,
    tau_h=24
):
    loading_dose = target_concentration * vd_l / bioavailability
    maintenance_dose = target_concentration * cl_l_h * tau_h / bioavailability

    summary = pd.DataFrame({
        "Metric": [
            "Target concentration",
            "Vd",
            "Bioavailability",
            "CL",
            "Tau",
            "Estimated loading dose",
            "Estimated maintenance dose per interval"
        ],
        "Value": [
            f"{target_concentration:.2f} mg/L",
            f"{vd_l:.1f} L",
            f"{bioavailability:.2f}",
            f"{cl_l_h:.2f} L/h",
            f"{tau_h:.1f} h",
            f"{loading_dose:.0f} mg",
            f"{maintenance_dose:.0f} mg"
        ]
    })

    display(summary)


interact(
    calculate_loading_and_maintenance,
    target_concentration=FloatSlider(value=5, min=1, max=20, step=0.5, description="Target C"),
    vd_l=FloatSlider(value=70, min=10, max=200, step=5, description="Vd"),
    bioavailability=FloatSlider(value=0.9, min=0.1, max=1.0, step=0.05, description="F"),
    cl_l_h=FloatSlider(value=8, min=0.5, max=20, step=0.5, description="CL"),
    tau_h=FloatSlider(value=24, min=6, max=48, step=6, description="Tau")
);

interactive(children=(FloatSlider(value=5.0, description='Target C', max=20.0, min=1.0, step=0.5), FloatSlider…

## 15. 观察任务 5：负荷剂量和维持剂量

请完成以下操作：

### 任务 A：改变 Vd

保持其他参数不变，将 Vd 从 70 L 改为 140 L。

观察：

- Estimated loading dose 是否增加？
- Estimated maintenance dose 是否一定同比例增加？

### 任务 B：改变 CL

将 Vd 恢复为 70 L，将 CL 从 8 L/h 改为 2 L/h。

观察：

- Estimated loading dose 是否明显变化？
- Estimated maintenance dose 是否下降？

### 任务 C：临床解释

思考：

- 为什么肾功能下降时，维持剂量通常需要调整？
- 为什么某些药物在肾功能下降时仍可能保留首剂或负荷剂量？

## 16. 自测题：肾功能与剂量调整

请根据本 Notebook 的内容完成以下自测题。建议先独立作答，再查看下一单元格中的参考答案。

---

### 题目 1：肾功能下降最直接影响哪一个 PK 参数？

A. Emax  
B. EC50  
C. CL  
D. Hill 系数  

---

### 题目 2：Cockcroft-Gault 公式主要用于估算什么？

A. 肝清除率  
B. 肌酐清除率 CrCl  
C. 药物最大效应 Emax  
D. 口服生物利用度 F  

---

### 题目 3：对于主要经肾脏清除的药物，CrCl 下降时最可能发生什么？

A. AUC 降低，半衰期缩短  
B. AUC 升高，半衰期延长  
C. Cmax 一定变为 0  
D. 药效一定完全消失  

---

### 题目 4：减少剂量和延长给药间隔相比，哪一项描述更合理？

A. 两者永远产生完全相同的浓度曲线  
B. 减少剂量通常降低峰浓度，延长间隔可能保留较高峰浓度但谷浓度更低  
C. 延长间隔一定会增加毒性  
D. 减少剂量一定会导致无效  

---

### 题目 5：肾功能下降时，为什么有些药物仍可能保留首剂或负荷剂量？

A. 因为负荷剂量主要由 Vd 和目标浓度决定  
B. 因为负荷剂量只由 CL 决定  
C. 因为肾功能下降会使 Vd 永远变为 0  
D. 因为所有药物都不需要调整维持剂量

## 17. 自测题参考答案

### 题目 1

**参考答案：C**

**解析：**  
肾功能下降最直接影响的是依赖肾脏排泄的药物清除率 CL。Emax、EC50 和 Hill 系数属于 PD 参数，不是肾功能下降最直接改变的 PK 参数。

---

### 题目 2

**参考答案：B**

**解析：**  
Cockcroft-Gault 公式用于估算肌酐清除率 CrCl。许多药物说明书中的肾功能剂量调整建议以 CrCl 分层。

---

### 题目 3

**参考答案：B**

**解析：**  
对于主要经肾脏清除的药物，CrCl 下降会使 CL 下降。根据：

$$
AUC = \frac{Dose}{CL}
$$

以及：

$$
t_{1/2} = \frac{0.693 \times V_d}{CL}
$$

CL 下降会导致 AUC 升高、半衰期延长，药物更容易蓄积。

---

### 题目 4

**参考答案：B**

**解析：**  
减少剂量和延长间隔都可以降低维持剂量速率，但浓度曲线形状不同。减少剂量通常降低峰浓度和波动；延长给药间隔可能保留较高峰浓度，但给药间隔末端的谷浓度更低。

---

### 题目 5

**参考答案：A**

**解析：**  
负荷剂量主要用于快速达到目标浓度，主要由目标浓度、Vd 和 F 决定。维持剂量用于补偿清除，主要与 CL 有关。因此肾功能下降时，更直接影响的是维持剂量或给药间隔。

## 18. 本 Notebook 小结

本 Notebook 通过肾功能变化和剂量调整模拟，帮助你理解临床药学中个体化给药的基本逻辑。

你应该掌握以下核心结论：

1. 肾功能下降主要影响以肾脏清除为主的药物。
2. 清除率降低会增加 AUC、延长半衰期，并可能增加药物蓄积风险。
3. 肾功能不全对药物暴露的影响，很大程度上取决于原形经尿排泄比例 $f_e$。
4. Cockcroft-Gault 公式估算的 CrCl 常用于药物剂量调整，但其本身存在临床局限性。
5. 降低剂量和延长给药间隔都可以降低暴露量，但二者对峰浓度和谷浓度的影响不同。
6. 负荷剂量主要由 $V_d$ 和目标浓度决定，而维持给药方案主要由清除率决定。


本节的完整逻辑可以概括为：

$$
Renal\ function \rightarrow CL \rightarrow AUC/t_{1/2} \rightarrow Accumulation\ risk \rightarrow Dose\ adjustment
$$

下一节将进一步进入抗菌药物 PK/PD 应用，例如：

> 万古霉素 AUC/MIC 模拟。

## 参考资料

1. Cockcroft DW, Gault MH. Prediction of creatinine clearance from serum creatinine. *Nephron*. 1976;16(1):31-41.  
   PubMed: https://pubmed.ncbi.nlm.nih.gov/1244564/

2. U.S. Food and Drug Administration. *Pharmacokinetics in Patients with Impaired Renal Function — Study Design, Data Analysis, and Impact on Dosing*. Final guidance, March 2024.  
   FDA: https://www.fda.gov/regulatory-information/search-fda-guidance-documents/pharmacokinetics-patients-impaired-renal-function-study-design-data-analysis-and-impact-dosing